# Лабораторна робота 2_2. Харченко, ФІТ 3-10

Аналіз двох наборів даних з Kaggle:
1. **Titanic** - дані про пасажирів Титаніка
2. **Amazon Top 50 Bestselling Books 2009-2019** - 50 найпродаваніших книг Amazon за кожен рік

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub

## Частина 1. Titanic

### 1. Завантаження даних
Датасет завантажуємо з Kaggle через бібліотеку kagglehub.

In [ ]:
path = kagglehub.dataset_download("mahmoudshogaa/titanic-dataset")
titanic = pd.read_csv(os.path.join(path, "titanic.csv"))
titanic.head()

Стовпці:
- Survived - вижив (1) чи ні (0)
- Pclass - клас квитка (1, 2, 3)
- Sex - стать, Age - вік
- SibSp - брати, сестри, чоловік або дружина на борту
- Parch - батьки або діти на борту
- Fare - ціна квитка, Cabin - каюта
- Embarked - порт посадки (S - Саутгемптон, C - Шербур, Q - Квінстаун)

### 2. Розмір і типи даних

In [ ]:
print("Рядків:", titanic.shape[0])
print("Стовпців:", titanic.shape[1])
titanic.info()

### 3. Дублікати

In [ ]:
# перевіряємо до видалення стовпців, бо без імені різні люди можуть мати однакові дані
print("Дублікатів:", titanic.duplicated().sum())

### 4. Видалення зайвих стовпців

In [ ]:
print("Пропусків у Cabin:", round(titanic["Cabin"].isna().mean() * 100, 1), "%")

# PassengerId, Name і Ticket в кожного свої, для аналізу нічого не дають
# Cabin пустий у більшості пасажирів, тому його теж видаляємо
titanic = titanic.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])
titanic.head()

### 5. Пропущені значення

In [ ]:
titanic.isna().sum()

In [ ]:
# вік заповнюємо медіаною, вона менше залежить від дуже старих або дуже малих пасажирів
titanic["Age"] = titanic["Age"].fillna(titanic["Age"].median())

# порт посадки заповнюємо найчастішим значенням
titanic["Embarked"] = titanic["Embarked"].fillna(titanic["Embarked"].mode()[0])

titanic.isna().sum()

### 6. Описова статистика

In [ ]:
titanic.describe()

### 7. Скільки пасажирів вижило

In [ ]:
print(titanic["Survived"].value_counts())
print("Вижило:", round(titanic["Survived"].mean() * 100, 1), "%")

### 8. Виживання за статтю

In [ ]:
by_sex = titanic.groupby("Sex")["Survived"].mean() * 100
print(by_sex.round(1))

by_sex.plot(kind="bar", color=["tab:red", "tab:blue"])
plt.title("Відсоток тих, хто вижив, за статтю")
plt.ylabel("%")
plt.xticks(rotation=0)
plt.show()

### 9. Виживання за класом

In [ ]:
by_class = titanic.groupby("Pclass")["Survived"].mean() * 100
print(by_class.round(1))

by_class.plot(kind="bar", color="tab:green")
plt.title("Відсоток тих, хто вижив, за класом")
plt.ylabel("%")
plt.xticks(rotation=0)
plt.show()

### 10. Стать і клас разом

In [ ]:
pd.pivot_table(titanic, values="Survived", index="Sex", columns="Pclass", aggfunc="mean").mul(100).round(1)

### 11. Вік

In [ ]:
plt.hist(titanic["Age"], bins=30)
plt.title("Розподіл віку пасажирів")
plt.xlabel("Вік")
plt.ylabel("Кількість")
plt.show()
# пік біля 28 років - це пропуски, які ми заповнили медіаною

In [ ]:
titanic["AgeGroup"] = pd.cut(titanic["Age"], bins=[0, 12, 18, 40, 60, 80],
                             labels=["0-12", "13-18", "19-40", "41-60", "61-80"])
age_table = titanic.groupby("AgeGroup", observed=True)["Survived"].agg(["count", "mean"])
age_table["mean"] = (age_table["mean"] * 100).round(1)
age_table.columns = ["Пасажирів", "Вижило, %"]
age_table

### 12. Ціна квитка

In [ ]:
print("Середня і медіанна ціна квитка за класом:")
print(titanic.groupby("Pclass")["Fare"].agg(["mean", "median"]).round(2))
print()
print("Середня ціна квитка (0 - загинули, 1 - вижили):")
print(titanic.groupby("Survived")["Fare"].mean().round(2))

### 13. Розмір сім'ї

In [ ]:
# сім'я = брати/сестри/подружжя + батьки/діти + сама людина
titanic["Family"] = titanic["SibSp"] + titanic["Parch"] + 1

family = titanic.groupby("Family")["Survived"].agg(["count", "mean"])
family["mean"] = (family["mean"] * 100).round(1)
family.columns = ["Пасажирів", "Вижило, %"]
print(family)

family["Вижило, %"].plot(kind="bar", color="tab:purple")
plt.title("Відсоток тих, хто вижив, за розміром сім'ї")
plt.xlabel("Людей у сім'ї")
plt.ylabel("%")
plt.xticks(rotation=0)
plt.show()

### 14. Порт посадки

In [ ]:
titanic.groupby("Embarked")["Survived"].agg(["count", "mean"]).round(3)

### 15. Кореляція

In [ ]:
num = titanic[["Survived", "Pclass", "Age", "SibSp", "Parch", "Fare", "Family"]].copy()
num["Sex"] = (titanic["Sex"] == "female").astype(int)  # 1 - жінка, 0 - чоловік

corr = num.corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Кореляція ознак, Titanic")
plt.show()

print("Кореляція з Survived:")
print(corr["Survived"].drop("Survived").sort_values(ascending=False).round(2))

### Висновки по Titanic
- вижило тільки 38,4% пасажирів (342 з 891)
- найсильніше на виживання впливала стать: вижило 74% жінок і тільки 19% чоловіків (кореляція 0.54)
- також дуже важливий клас: у 1 класі вижило 63%, у 3 класі тільки 24%. Жінки з 1 і 2 класу вижили майже всі (97% і 92%)
- діти до 12 років виживали частіше за інших (58%), а пасажири старші 60 найрідше (23%)
- ті, хто вижив, в середньому платили за квиток більше (48 проти 22), це пов'язано з класом
- найкраще виживали сім'ї з 2-4 людей, а пасажири без сім'ї і великі сім'ї (5 і більше людей) виживали гірше
- пасажири, які сіли в Шербурі (C), вижили частіше (55%), мабуть тому, що серед них було багато пасажирів 1 класу

## Частина 2. Amazon Top 50 Bestselling Books 2009-2019

### 1. Завантаження даних

In [ ]:
path = kagglehub.dataset_download("sootersaalu/amazon-top-50-bestselling-books-2009-2019")
books = pd.read_csv(os.path.join(path, "bestsellers with categories.csv"))
books.head()

Стовпці: Name - назва книги, Author - автор, User Rating - рейтинг (до 5), Reviews - кількість відгуків, Price - ціна в доларах, Year - рік, Genre - жанр (Fiction - художня, Non Fiction - нехудожня).

### 2. Розмір і типи даних

In [ ]:
print("Рядків:", books.shape[0])
print("Стовпців:", books.shape[1])
books.info()

### 3. Пропуски і дублікати

In [ ]:
print("Пропусків:")
print(books.isna().sum())
print()
print("Повних дублікатів:", books.duplicated().sum())
print("Унікальних книг:", books["Name"].nunique(), "з", len(books))
# одна книга може бути в топі кілька років підряд, це не дублікат, тому нічого не видаляємо

### 4. Очистка даних

In [ ]:
# один і той самий автор записаний по-різному
print(books.loc[books["Author"].str.contains("Martin|Rowling"), "Author"].unique())

books["Author"] = books["Author"].replace({"George R. R. Martin": "George R.R. Martin",
                                           "J. K. Rowling": "J.K. Rowling"})
print(books.loc[books["Author"].str.contains("Martin|Rowling"), "Author"].unique())

In [ ]:
# книги з ціною 0
print("Книг з ціною 0:", (books["Price"] == 0).sum())
books[books["Price"] == 0][["Name", "Author", "Year"]]

### 5. Описова статистика

In [ ]:
books.describe()

### 6. Жанри

In [ ]:
print(books["Genre"].value_counts())

genre_year = pd.crosstab(books["Year"], books["Genre"])
genre_year.plot(kind="bar", figsize=(10, 5))
plt.title("Кількість книг кожного жанру в топ-50 по роках")
plt.ylabel("Книг")
plt.xticks(rotation=0)
plt.show()

### 7. Ціна

In [ ]:
print("Ціна за жанром:")
print(books.groupby("Genre")["Price"].agg(["mean", "median", "max"]).round(2))

books.groupby("Year")["Price"].mean().plot(marker="o")
plt.title("Середня ціна книг у топі по роках")
plt.ylabel("Ціна, $")
plt.show()

In [ ]:
# найдорожчі книги (кожну книгу беремо один раз)
books.drop_duplicates("Name").nlargest(5, "Price")[["Name", "Author", "Price", "Genre"]]

### 8. Рейтинг

In [ ]:
print("Середній рейтинг за жанром:")
print(books.groupby("Genre")["User Rating"].mean().round(3))

plt.hist(books["User Rating"], bins=15)
plt.title("Розподіл рейтингу книг")
plt.xlabel("Рейтинг")
plt.ylabel("Кількість")
plt.show()

In [ ]:
# книги з найнижчим рейтингом
books.drop_duplicates("Name").nsmallest(5, "User Rating")[["Name", "Author", "User Rating", "Year"]]

### 9. Автори

In [ ]:
top_authors = books["Author"].value_counts().head(10)
print(top_authors)

top_authors.sort_values().plot(kind="barh", color="tab:orange")
plt.title("Автори, які найчастіше потрапляли в топ-50")
plt.xlabel("Разів у топі")
plt.show()

### 10. Книги з найбільшою кількістю відгуків

In [ ]:
books.drop_duplicates("Name").nlargest(10, "Reviews")[["Name", "Author", "Reviews", "Genre"]]

### 11. Кореляція

In [ ]:
corr_books = books[["User Rating", "Reviews", "Price", "Year"]].corr()
sns.heatmap(corr_books, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Кореляція ознак, Amazon")
plt.show()

### Висновки по Amazon
- у таблиці 550 записів, але тільки 351 унікальна книга, бо багато книг були в топі кілька років
- нехудожніх книг (Non Fiction) у топі більше: 310 проти 240. Тільки у 2014 художніх було більше (29 з 50)
- нехудожні книги в середньому дорожчі (14,8$ проти 10,9$), найдорожча книга - DSM-5 за 105$. Є 12 записів з ціною 0
- середня ціна книг у топі з роками падає (з 15,4$ у 2009 до 10,1$ у 2019), а середній рейтинг навпаки росте (з 4,58 до 4,74)
- рейтинги загалом дуже високі (в середньому 4,6), найнижчий у The Casual Vacancy від J.K. Rowling (3,3)
- найчастіше в топ потрапляв Jeff Kinney (Diary of a Wimpy Kid), 12 разів
- найбільше відгуків у книги Where the Crawdads Sing, майже 88 тисяч
- сильних кореляцій між рейтингом, кількістю відгуків і ціною немає (всі коефіцієнти менше 0,3 за модулем), тобто дорожча книга не означає вищий рейтинг